# Análise do Mercado de Trabalho Formal — CAGED & RAIS
## Pipeline Completo: Extração, Processamento e Análise
### Estados do Nordeste (2015–2025)

---

Este notebook apresenta o fluxo completo de trabalho com dados do **CAGED** e da **RAIS**, desde a
extração dos microdados no servidor FTP do Ministério do Trabalho e Emprego (MTE/PDET) até a análise
e visualização dos resultados.

### Fontes de dados

| Base | Cobertura | Arquivo no FTP |
|------|-----------|----------------|
| CAGED Antigo | 2015 – 2019 (pré-eSocial) | `CAGED/{ano}/CAGEDEST_{MMAA}.7z` |
| Novo CAGED   | 2020 – 2025 (pós-eSocial) | `NOVO CAGED/{ano}/{anoMM}/CAGEDMOV{anoMM}.7z` |
| RAIS         | 2015 – 2023              | `RAIS/{ano}/RAIS_VINC_PUB_NORDESTE.7z` |

**FTP base:** `ftp://ftp.mtps.gov.br/pdet/microdados/`

### Datasets produzidos pela pipeline

| Arquivo | Descrição |
|---------|-----------|
| `caged_antigo_saldo_mensal.csv` | Saldo mensal por UF (2015-2019) |
| `caged_antigo_por_setor.csv` | Saldo anual por UF e seção CNAE (2015-2019) |
| `caged_antigo_por_perfil.csv` | Saldo por UF, sexo e escolaridade (2015-2019) |
| `caged_saldo_mensal.csv` | Saldo mensal por UF (2020-2025) |
| `caged_por_setor.csv` | Saldo anual por UF e seção CNAE (2020-2025) |
| `caged_por_perfil.csv` | Saldo por UF, sexo e escolaridade (2020-2025) |
| `rais_vinculos.csv` | Vínculos ativos e remuneração por UF/ano |
| `rais_por_setor.csv` | Vínculos ativos por UF, ano e divisão CNAE |

### Estrutura do notebook

```
1. Configuração do Ambiente
2. Pipeline de Extração (FTP → CSV)
   2.1 Funções auxiliares (FTP, descompactação 7z, normalização)
   2.2 CAGED Antigo (2015-2019)
   2.3 Novo CAGED (2020-2025)
   2.4 RAIS
3. Análise Exploratória
   3.1 Carregamento e unificação das séries
   3.2 Panorama geral — saldo de emprego formal
   3.3 Saldo acumulado anual por estado
   3.4 Heatmap — saldo mensal por estado
   3.5 Evolução do salário médio
   3.6 Impacto da COVID-19 (2020)
   3.7 Análise por setor econômico
   3.8 Perfil — admissões por sexo e escolaridade
```

> ⚠️ **Atenção:** A etapa de extração (Seção 2) realiza downloads de arquivos grandes via FTP.
> Dependendo da conexão, pode levar **horas**. Se os arquivos CSV já estiverem disponíveis
> localmente, as funções detectam o cache automaticamente e pulam o download.

---
## 1. Configuração do Ambiente

### Instalação das dependências

In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet',
                       'py7zr', 'pandas', 'matplotlib', 'seaborn'])
print('Dependências instaladas.')

Dependências instaladas.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip


### Configuração dos caminhos de saída

In [2]:
from pathlib import Path

RAW_DIR = Path('processed')

PERIODO_INICIO = 2015
PERIODO_FIM    = 2025

ESTADOS_NE = {
    'AL': {'cod_ibge': '27'},
    'BA': {'cod_ibge': '29'},
    'CE': {'cod_ibge': '23'},
    'MA': {'cod_ibge': '21'},
    'PB': {'cod_ibge': '25'},
    'PE': {'cod_ibge': '26'},
    'PI': {'cod_ibge': '22'},
    'RN': {'cod_ibge': '24'},
    'SE': {'cod_ibge': '28'},
}

for sub in ['caged/nordeste', 'rais/nordeste', 'coleta_status']:
    (RAW_DIR / sub).mkdir(parents=True, exist_ok=True)

print(f'Diretório de dados: {RAW_DIR.resolve()}')
print(f'Período: {PERIODO_INICIO} – {PERIODO_FIM}')
print(f'Estados: {list(ESTADOS_NE.keys())}')

Diretório de dados: /Users/cassiopinheiro/dev/academico/bruno/notebook/processed
Período: 2015 – 2025
Estados: ['AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE']


---
## 2. Pipeline de Extração (FTP → CSV)

> Esta seção implementa o download dos microdados diretamente do servidor FTP do MTE/PDET,
> descompacta os arquivos `.7z` e gera CSVs agregados por UF.

### 2.1 — Funções Auxiliares

In [3]:
import logging
import tempfile
from ftplib import FTP, error_perm

import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    datefmt='%H:%M:%S'
)
logger = logging.getLogger(__name__)

FTP_HOST              = 'ftp.mtps.gov.br'
CAGED_ANTIGO_FTP_BASE = '/pdet/microdados/CAGED'
CAGED_FTP_BASE        = '/pdet/microdados/NOVO CAGED'
RAIS_FTP_BASE         = '/pdet/microdados/RAIS'

UFS_NE_IBGE   = {int(info['cod_ibge']) for info in ESTADOS_NE.values()}
UF_IBGE_SIGLA = {int(info['cod_ibge']): uf for uf, info in ESTADOS_NE.items()}

print('Configuração carregada!')
print(f'UFs Nordeste (cód. IBGE): {sorted(UFS_NE_IBGE)}')

Configuração carregada!
UFs Nordeste (cód. IBGE): [21, 22, 23, 24, 25, 26, 27, 28, 29]


In [4]:
def _ftp_download(remote_path: str, local_path: str):
    """Baixa um arquivo do FTP do MTE para o disco local."""
    logger.info(f'FTP download: {FTP_HOST}{remote_path}')
    with FTP(FTP_HOST, timeout=120) as ftp:
        ftp.login()
        with open(local_path, 'wb') as f:
            ftp.retrbinary(f'RETR {remote_path}', f.write)
    logger.info(f'Download concluído: {Path(local_path).name}')


def _read_7z(archive_path: str, extract_dir: str,
             sep: str = ';', encoding: str = 'latin-1') -> pd.DataFrame:
    """Extrai o arquivo .7z para um diretório temporário e lê o primeiro CSV/TXT."""
    import py7zr

    with py7zr.SevenZipFile(archive_path, 'r') as z:
        names = z.getnames()
        data_file = next((n for n in names if n.lower().endswith(('.csv', '.txt'))), None)
        if data_file is None:
            raise ValueError(f'Nenhum CSV/TXT em {archive_path}: {names}')
        logger.info(f'Extraindo {data_file}...')
        z.extractall(path=extract_dir)

    file_path = Path(extract_dir) / data_file
    try:
        return pd.read_csv(file_path, sep=sep, encoding=encoding, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(file_path, sep=sep, encoding='utf-8', low_memory=False)


def _fix_mojibake(text: str) -> str:
    """Corrige mojibake (UTF-8 lido como latin-1) em nomes de colunas."""
    try:
        return text.encode('latin-1').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        return text


def _filtrar_nordeste(df: pd.DataFrame) -> pd.DataFrame:
    """Filtra registros do Nordeste por UF ou região."""
    if 'uf' in df.columns:
        df['uf'] = pd.to_numeric(df['uf'], errors='coerce')
        return df[df['uf'].isin(UFS_NE_IBGE)].copy()
    if 'regiao' in df.columns:
        df['regiao'] = pd.to_numeric(df['regiao'], errors='coerce')
        return df[df['regiao'] == 2].copy()
    return df


def _all_paths_exist(paths: list) -> bool:
    """Retorna True somente se todos os arquivos listados existirem no disco."""
    return all(Path(p).exists() for p in paths)


def _save_dataframe(df: pd.DataFrame, filename: str, path_parts: list):
    """Salva um DataFrame como CSV no subdiretório indicado."""
    out_dir = RAW_DIR / '/'.join(path_parts)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f'{filename}.csv'
    df.to_csv(out_path, index=False)
    logger.info(f'Salvo: {out_path}')
    return out_path


def _save_status_manifest(rows: list, filename: str):
    """Persiste o manifesto de status da coleta para auditoria."""
    if not rows:
        return
    df = pd.DataFrame(rows)
    sort_cols = [c for c in ['ano', 'mes', 'uf', 'arquivo'] if c in df.columns]
    df.sort_values(sort_cols, inplace=True)
    _save_dataframe(df, filename, path_parts=['coleta_status'])


print('Funções auxiliares definidas!')

Funções auxiliares definidas!


In [5]:
def _normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normaliza os nomes de colunas do Novo CAGED (pós-2020),
    tratando variações de layout entre os anos.
    """
    logger.info(f'Colunas originais: {list(df.columns)}')
    df.columns = [_fix_mojibake(c) for c in df.columns]

    lower_cols = {col: col.lower().strip() for col in df.columns}
    has_valor_salario   = any(lc in ('valorsaláriofixo', 'valorsalariofixo') for lc in lower_cols.values())
    has_competencia_mov = any('competenciamov' in lc or 'competênciamov' in lc for lc in lower_cols.values())

    mapping = {}
    for col in df.columns:
        lc = lower_cols[col]
        if 'competenciamov' in lc or 'competênciamov' in lc:
            mapping[col] = 'competencia'
        elif ('competencia' in lc or 'competência' in lc) and not has_competencia_mov and 'competencia' not in mapping.values():
            mapping[col] = 'competencia'
        elif lc == 'uf':
            mapping[col] = 'uf'
        elif 'municipio' in lc or 'município' in lc:
            mapping[col] = 'municipio'
        elif 'saldomov' in lc:
            mapping[col] = 'saldo_movimentacao'
        elif lc in ('valorsaláriofixo', 'valorsalariofixo'):
            mapping[col] = 'salario'
        elif lc in ('salário', 'salario', 'salmensal') and not has_valor_salario:
            mapping[col] = 'salario'
        elif 'grau' in lc and 'instru' in lc:
            mapping[col] = 'grau_instrucao'
        elif lc == 'sexo':
            mapping[col] = 'sexo'
        elif 'raca' in lc or 'raça' in lc:
            mapping[col] = 'raca_cor'
        elif lc == 'idade':
            mapping[col] = 'idade'
        elif 'cbo' in lc and 'ocupa' in lc:
            mapping[col] = 'cbo_2002'
        elif lc == 'subclasse':
            mapping[col] = 'cnae_subclasse'
        elif lc in ('seção', 'secao', 'seçao'):
            mapping[col] = 'secao_cnae'
        elif lc in ('região', 'regiao'):
            mapping[col] = 'regiao'

    logger.info(f'Mapeamento: {mapping}')
    return df.rename(columns=mapping)


print('Função de normalização de colunas definida!')

Função de normalização de colunas definida!


### 2.2 — CAGED Antigo (2015–2019)

O **CAGED Antigo** usa o layout pré-eSocial. Os microdados são publicados mensalmente com a
competência no formato `MMAA` (ex.: `012015` = janeiro de 2015).

**Agregações geradas:**
- `caged_antigo_saldo_mensal.csv` — saldo de admissões e desligamentos por UF e mês
- `caged_antigo_por_setor.csv`    — saldo anual por UF e divisão CNAE
- `caged_antigo_por_perfil.csv`   — saldo por UF, sexo e grau de instrução

In [6]:
def _processar_caged_antigo_mes(ano: int, mes: int) -> pd.DataFrame | None:
    """
    Baixa e processa um mês do CAGED Antigo (pré-2020).
    Retorna DataFrame filtrado para o Nordeste, ou None se o arquivo
    não estiver disponível no FTP.
    """
    competencia = f'{mes:02d}{ano}'
    remote_path = f'{CAGED_ANTIGO_FTP_BASE}/{ano}/CAGEDEST_{competencia}.7z'

    with tempfile.TemporaryDirectory() as tmpdir:
        local_7z = Path(tmpdir) / f'CAGEDEST_{competencia}.7z'

        try:
            _ftp_download(remote_path, str(local_7z))
        except error_perm as e:
            logger.warning(f'CAGED Antigo {ano}/{mes:02d}: não disponível ({e})')
            return None
        except Exception as e:
            logger.warning(f'CAGED Antigo {ano}/{mes:02d}: erro download ({e})')
            return None

        try:
            df = _read_7z(str(local_7z), tmpdir)
        except Exception as e:
            logger.error(f'CAGED Antigo {ano}/{mes:02d}: erro ao extrair ({e})')
            return None

    df.columns = [_fix_mojibake(c) for c in df.columns]
    col_map = {}
    for col in df.columns:
        lc = col.lower().strip()
        if lc in ('município', 'municipio'):                                      col_map[col] = 'municipio'
        elif lc == 'uf':                                                           col_map[col] = 'uf'
        elif lc in ('região', 'regiao'):                                           col_map[col] = 'regiao'
        elif lc in ('admitidos/desligados', 'admitidosdesligados', 'tipo_mov'):   col_map[col] = 'admitidos_desligados'
        elif 'salario' in lc or 'salário' in lc or lc == 'salmensal':             col_map[col] = 'salario'
        elif 'grau' in lc and 'instru' in lc:                                     col_map[col] = 'grau_instrucao'
        elif lc == 'sexo':                                                         col_map[col] = 'sexo'
        elif 'raca' in lc or 'raça' in lc:                                        col_map[col] = 'raca_cor'
        elif lc in ('subatividade ibge', 'subclasse', 'cnae20subclasse', 'cnae_2_subclasse'): col_map[col] = 'cnae_subclasse'
        elif lc in ('seção', 'secao', 'seçao'):                                   col_map[col] = 'secao_cnae'
    df.rename(columns=col_map, inplace=True)

    if 'grau_instrucao' in df.columns:
        df['grau_instrucao'] = pd.to_numeric(df['grau_instrucao'].astype(str).str.strip(), errors='coerce')
    if 'sexo' in df.columns:
        df['sexo'] = pd.to_numeric(df['sexo'], errors='coerce')

    df = _filtrar_nordeste(df)
    if df.empty:
        logger.warning(f'CAGED Antigo {ano}/{mes:02d}: sem dados Nordeste.')
        return None

    if 'admitidos_desligados' in df.columns:
        df['admitidos_desligados'] = pd.to_numeric(df['admitidos_desligados'], errors='coerce')
        df['saldo_movimentacao']   = df['admitidos_desligados'].map({1: 1, 2: -1, 3: -1})
        df['saldo_movimentacao']   = df['saldo_movimentacao'].fillna(0).astype(int)

    if 'uf' in df.columns:
        df['uf']       = pd.to_numeric(df['uf'], errors='coerce')
        df['sigla_uf'] = df['uf'].map(UF_IBGE_SIGLA)

    df['ano'] = ano
    df['mes'] = mes

    if 'salario' in df.columns:
        df['salario'] = pd.to_numeric(
            df['salario'].astype(str).str.replace(',', '.', regex=False), errors='coerce'
        )

    logger.info(f'CAGED Antigo {ano}/{mes:02d}: {len(df)} registros Nordeste.')
    return df


print('Função _processar_caged_antigo_mes definida!')

Função _processar_caged_antigo_mes definida!


In [ ]:
def coletar_caged_antigo_nordeste(ano_inicio: int = PERIODO_INICIO, ano_fim: int = 2019):
    """
    Coleta o CAGED Antigo (2015–2019) e gera 3 CSVs agregados.
    Se os arquivos já existirem no disco, pula o download (cache).
    """
    outputs = [
        RAW_DIR / 'caged' / 'nordeste' / 'caged_antigo_saldo_mensal.csv',
        RAW_DIR / 'caged' / 'nordeste' / 'caged_antigo_por_setor.csv',
        RAW_DIR / 'caged' / 'nordeste' / 'caged_antigo_por_perfil.csv',
    ]
    if _all_paths_exist(outputs):
        logger.info('CAGED Antigo Nordeste: cache encontrado, pulando coleta.')
        return pd.read_csv(outputs[0])

    saldo_frames, setor_frames, perfil_frames, status_rows = [], [], [], []

    for ano in range(ano_inicio, ano_fim + 1):
        for mes in range(1, 13):
            df = _processar_caged_antigo_mes(ano, mes)
            if df is None:
                status_rows.append({'fonte': 'caged_antigo', 'ano': ano, 'mes': mes, 'status': 'falha_ou_ausente'})
                continue
            status_rows.append({'fonte': 'caged_antigo', 'ano': ano, 'mes': mes, 'status': 'ok', 'registros': len(df)})

            if 'sigla_uf' in df.columns and 'saldo_movimentacao' in df.columns:
                agg_kwargs = dict(admissoes=('saldo_movimentacao', lambda x: (x == 1).sum()),
                                  desligamentos=('saldo_movimentacao', lambda x: (x == -1).sum()),
                                  saldo=('saldo_movimentacao', 'sum'),
                                  total_movimentacoes=('saldo_movimentacao', 'count'))
                if 'salario' in df.columns:
                    agg_kwargs['salario_medio'] = ('salario', 'mean')
                agg1 = df.groupby(['ano', 'mes', 'sigla_uf'], as_index=False).agg(**agg_kwargs)
                saldo_frames.append(agg1)

            cnae_col = ('secao_cnae' if 'secao_cnae' in df.columns
                        else 'cnae_subclasse' if 'cnae_subclasse' in df.columns else None)
            if cnae_col and 'sigla_uf' in df.columns:
                df['divisao_cnae'] = (df[cnae_col].astype(str).str[:2]
                                      if cnae_col == 'cnae_subclasse'
                                      else df[cnae_col].astype(str))
                agg_kwargs2 = dict(saldo=('saldo_movimentacao', 'sum'),
                                   total_movimentacoes=('saldo_movimentacao', 'count'))
                if 'salario' in df.columns:
                    agg_kwargs2['salario_medio'] = ('salario', 'mean')
                agg2 = df.groupby(['ano', 'sigla_uf', 'divisao_cnae'], as_index=False).agg(**agg_kwargs2)
                setor_frames.append(agg2)

            if all(c in df.columns for c in ('sigla_uf', 'sexo', 'grau_instrucao')):
                agg_kwargs3 = dict(admissoes=('saldo_movimentacao', lambda x: (x == 1).sum()),
                                   desligamentos=('saldo_movimentacao', lambda x: (x == -1).sum()),
                                   saldo=('saldo_movimentacao', 'sum'),
                                   total_movimentacoes=('saldo_movimentacao', 'count'))
                if 'salario' in df.columns:
                    agg_kwargs3['salario_medio'] = ('salario', 'mean')
                agg3 = df.groupby(['ano', 'sigla_uf', 'sexo', 'grau_instrucao'], as_index=False).agg(**agg_kwargs3)
                perfil_frames.append(agg3)

    result = None
    if saldo_frames:
        df_s = pd.concat(saldo_frames, ignore_index=True).sort_values(['ano', 'mes', 'sigla_uf'])
        _save_dataframe(df_s, 'caged_antigo_saldo_mensal', path_parts=['caged', 'nordeste'])
        result = df_s
        logger.info(f'CAGED Antigo saldo mensal: {len(df_s)} linhas salvas.')
    if setor_frames:
        df_setor = pd.concat(setor_frames, ignore_index=True)
        df_setor = df_setor.groupby(['ano', 'sigla_uf', 'divisao_cnae'], as_index=False).agg(
            saldo=('saldo', 'sum'), salario_medio=('salario_medio', 'mean'),
            total_movimentacoes=('total_movimentacoes', 'sum'))
        _save_dataframe(df_setor.sort_values(['ano', 'sigla_uf', 'divisao_cnae']),
                        'caged_antigo_por_setor', path_parts=['caged', 'nordeste'])
    if perfil_frames:
        df_perf = pd.concat(perfil_frames, ignore_index=True)
        df_perf = df_perf.groupby(['ano', 'sigla_uf', 'sexo', 'grau_instrucao'], as_index=False).agg(
            admissoes=('admissoes', 'sum'), desligamentos=('desligamentos', 'sum'),
            saldo=('saldo', 'sum'), salario_medio=('salario_medio', 'mean'),
            total_movimentacoes=('total_movimentacoes', 'sum'))
        _save_dataframe(df_perf.sort_values(['ano', 'sigla_uf', 'sexo', 'grau_instrucao']),
                        'caged_antigo_por_perfil', path_parts=['caged', 'nordeste'])
    if result is None:
        logger.error('CAGED Antigo: nenhum dado coletado.')
    _save_status_manifest(status_rows, 'caged_antigo_status')
    return result


df_antigo = coletar_caged_antigo_nordeste()

15:20:33 | INFO | FTP download: ftp.mtps.gov.br/pdet/microdados/CAGED/2015/CAGEDEST_012015.7z
15:21:49 | INFO | Download concluído: CAGEDEST_012015.7z
15:21:52 | INFO | Extraindo CAGEDEST_012015.txt...
15:21:52 | ERROR | CAGED Antigo 2015/01: erro ao extrair (Corrupt input data)
15:21:52 | INFO | FTP download: ftp.mtps.gov.br/pdet/microdados/CAGED/2015/CAGEDEST_022015.7z
15:22:45 | INFO | Download concluído: CAGEDEST_022015.7z
15:22:45 | INFO | Extraindo CAGEDEST_022015.txt...
15:22:56 | INFO | CAGED Antigo 2015/02: 431570 registros Nordeste.
15:22:56 | INFO | FTP download: ftp.mtps.gov.br/pdet/microdados/CAGED/2015/CAGEDEST_032015.7z
15:23:32 | INFO | Download concluído: CAGEDEST_032015.7z
15:23:32 | INFO | Extraindo CAGEDEST_032015.txt...
15:23:32 | ERROR | CAGED Antigo 2015/03: erro ao extrair (Corrupt input data)
15:23:33 | INFO | FTP download: ftp.mtps.gov.br/pdet/microdados/CAGED/2015/CAGEDEST_042015.7z
15:25:14 | INFO | Download concluído: CAGEDEST_042015.7z
15:25:14 | INFO | Ex

### 2.3 — Novo CAGED (2020–2025)

O **Novo CAGED** entrou em vigor em janeiro de 2020 com o eSocial. O layout das colunas
mudou em relação ao CAGED Antigo (ex.: `SaldoMovimentacao`, `ValorSaláriofixo`).

**Convenção de saldo:** `+1` = admissão, `-1` = desligamento.

In [ ]:
def _processar_caged_mes(ano: int, mes: int) -> pd.DataFrame | None:
    """
    Baixa e processa um mês do Novo CAGED (2020+).
    Retorna DataFrame filtrado para o Nordeste, ou None se indisponível.
    """
    competencia = f'{ano}{mes:02d}'
    remote_path = f'{CAGED_FTP_BASE}/{ano}/{competencia}/CAGEDMOV{competencia}.7z'

    with tempfile.TemporaryDirectory() as tmpdir:
        local_7z = Path(tmpdir) / f'CAGEDMOV{competencia}.7z'

        try:
            _ftp_download(remote_path, str(local_7z))
        except error_perm as e:
            logger.warning(f'CAGED {competencia}: arquivo não disponível ({e})')
            return None
        except Exception as e:
            logger.warning(f'CAGED {competencia}: erro no download ({e})')
            return None

        try:
            df = _read_7z(str(local_7z), tmpdir)
        except Exception as e:
            logger.error(f'CAGED {competencia}: erro ao extrair ({e})')
            return None

    df = _normalize_columns(df)
    df = _filtrar_nordeste(df)
    if df.empty:
        logger.warning(f'CAGED {competencia}: sem dados Nordeste.')
        return None

    if 'uf' in df.columns:
        df['sigla_uf'] = df['uf'].map(UF_IBGE_SIGLA)
    df['ano'] = ano
    df['mes'] = mes

    if 'saldo_movimentacao' in df.columns:
        df['saldo_movimentacao'] = pd.to_numeric(df['saldo_movimentacao'], errors='coerce')
    if 'salario' in df.columns:
        df['salario'] = pd.to_numeric(
            df['salario'].astype(str).str.replace(',', '.', regex=False), errors='coerce')

    if 'sexo' in df.columns:
        df['sexo'] = pd.to_numeric(df['sexo'], errors='coerce').replace({3: 2})
    if 'grau_instrucao' in df.columns:
        df['grau_instrucao'] = pd.to_numeric(df['grau_instrucao'], errors='coerce')

    logger.info(f'CAGED {competencia}: {len(df)} registros Nordeste.')
    return df


print('Função _processar_caged_mes definida!')

Função _processar_caged_mes definida!


In [ ]:
def coletar_caged_nordeste(ano_inicio: int = 2020, ano_fim: int = PERIODO_FIM):
    """
    Coleta o Novo CAGED (2020+) e gera 3 CSVs agregados.
    Se os arquivos já existirem no disco, pula o download (cache).
    """
    outputs = [
        RAW_DIR / 'caged' / 'nordeste' / 'caged_saldo_mensal.csv',
        RAW_DIR / 'caged' / 'nordeste' / 'caged_por_setor.csv',
        RAW_DIR / 'caged' / 'nordeste' / 'caged_por_perfil.csv',
    ]
    if _all_paths_exist(outputs):
        logger.info('CAGED Nordeste: cache encontrado, pulando coleta.')
        return pd.read_csv(outputs[0])

    saldo_frames, setor_frames, perfil_frames, status_rows = [], [], [], []

    for ano in range(ano_inicio, ano_fim + 1):
        for mes in range(1, 13):
            df = _processar_caged_mes(ano, mes)
            if df is None:
                status_rows.append({'fonte': 'caged_novo', 'ano': ano, 'mes': mes, 'status': 'falha_ou_ausente'})
                continue
            status_rows.append({'fonte': 'caged_novo', 'ano': ano, 'mes': mes, 'status': 'ok', 'registros': len(df)})

            if 'sigla_uf' in df.columns and 'saldo_movimentacao' in df.columns:
                agg_kwargs = dict(admissoes=('saldo_movimentacao', lambda x: (x == 1).sum()),
                                  desligamentos=('saldo_movimentacao', lambda x: (x == -1).sum()),
                                  saldo=('saldo_movimentacao', 'sum'),
                                  total_movimentacoes=('saldo_movimentacao', 'count'))
                if 'salario' in df.columns:
                    agg_kwargs['salario_medio'] = ('salario', 'mean')
                agg1 = df.groupby(['ano', 'mes', 'sigla_uf'], as_index=False).agg(**agg_kwargs)
                saldo_frames.append(agg1)

            cnae_col = ('secao_cnae' if 'secao_cnae' in df.columns
                        else 'cnae_subclasse' if 'cnae_subclasse' in df.columns else None)
            if cnae_col and 'sigla_uf' in df.columns:
                df['divisao_cnae'] = (df[cnae_col].astype(str).str[:2]
                                      if cnae_col == 'cnae_subclasse'
                                      else df[cnae_col].astype(str))
                agg_kwargs2 = dict(saldo=('saldo_movimentacao', 'sum'),
                                   total_movimentacoes=('saldo_movimentacao', 'count'))
                if 'salario' in df.columns:
                    agg_kwargs2['salario_medio'] = ('salario', 'mean')
                agg2 = df.groupby(['ano', 'sigla_uf', 'divisao_cnae'], as_index=False).agg(**agg_kwargs2)
                setor_frames.append(agg2)

            if all(c in df.columns for c in ('sigla_uf', 'sexo', 'grau_instrucao')):
                agg_kwargs3 = dict(admissoes=('saldo_movimentacao', lambda x: (x == 1).sum()),
                                   desligamentos=('saldo_movimentacao', lambda x: (x == -1).sum()),
                                   saldo=('saldo_movimentacao', 'sum'),
                                   total_movimentacoes=('saldo_movimentacao', 'count'))
                if 'salario' in df.columns:
                    agg_kwargs3['salario_medio'] = ('salario', 'mean')
                agg3 = df.groupby(['ano', 'sigla_uf', 'sexo', 'grau_instrucao'], as_index=False).agg(**agg_kwargs3)
                perfil_frames.append(agg3)

    result = None
    if saldo_frames:
        df_s = pd.concat(saldo_frames, ignore_index=True).sort_values(['ano', 'mes', 'sigla_uf'])
        _save_dataframe(df_s, 'caged_saldo_mensal', path_parts=['caged', 'nordeste'])
        result = df_s
        logger.info(f'CAGED saldo mensal: {len(df_s)} linhas salvas.')
    if setor_frames:
        df_setor = pd.concat(setor_frames, ignore_index=True)
        df_setor = df_setor.groupby(['ano', 'sigla_uf', 'divisao_cnae'], as_index=False).agg(
            saldo=('saldo', 'sum'), salario_medio=('salario_medio', 'mean'),
            total_movimentacoes=('total_movimentacoes', 'sum'))
        _save_dataframe(df_setor.sort_values(['ano', 'sigla_uf', 'divisao_cnae']),
                        'caged_por_setor', path_parts=['caged', 'nordeste'])
    if perfil_frames:
        df_perf = pd.concat(perfil_frames, ignore_index=True)
        df_perf = df_perf.groupby(['ano', 'sigla_uf', 'sexo', 'grau_instrucao'], as_index=False).agg(
            admissoes=('admissoes', 'sum'), desligamentos=('desligamentos', 'sum'),
            saldo=('saldo', 'sum'), salario_medio=('salario_medio', 'mean'),
            total_movimentacoes=('total_movimentacoes', 'sum'))
        _save_dataframe(df_perf.sort_values(['ano', 'sigla_uf', 'sexo', 'grau_instrucao']),
                        'caged_por_perfil', path_parts=['caged', 'nordeste'])
    if result is None:
        logger.error('CAGED: nenhum dado coletado.')
    _save_status_manifest(status_rows, 'caged_novo_status')
    return result


df_novo = coletar_caged_nordeste()

16:04:09 | INFO | FTP download: ftp.mtps.gov.br/pdet/microdados/NOVO CAGED/2020/202001/CAGEDMOV202001.7z
16:05:03 | INFO | Download concluído: CAGEDMOV202001.7z
16:05:03 | INFO | Extraindo CAGEDMOV202001.txt...
16:05:07 | INFO | Colunas originais: ['competÃªnciamov', 'regiÃ£o', 'uf', 'municÃ\xadpio', 'seÃ§Ã£o', 'subclasse', 'saldomovimentaÃ§Ã£o', 'cbo2002ocupaÃ§Ã£o', 'categoria', 'graudeinstruÃ§Ã£o', 'idade', 'horascontratuais', 'raÃ§acor', 'sexo', 'tipoempregador', 'tipoestabelecimento', 'tipomovimentaÃ§Ã£o', 'tipodedeficiÃªncia', 'indtrabintermitente', 'indtrabparcial', 'salÃ¡rio', 'tamestabjan', 'indicadoraprendiz', 'origemdainformaÃ§Ã£o', 'competÃªnciadec', 'indicadordeforadoprazo', 'unidadesalÃ¡riocÃ³digo', 'valorsalÃ¡riofixo']
16:05:07 | INFO | Mapeamento: {'competênciamov': 'competencia', 'região': 'regiao', 'uf': 'uf', 'município': 'municipio', 'seção': 'secao_cnae', 'subclasse': 'cnae_subclasse', 'saldomovimentação': 'saldo_movimentacao', 'cbo2002ocupação': 'cbo_2002', 'graude

### 2.4 — RAIS (Relação Anual de Informações Sociais)

A RAIS é um censo anual do mercado de trabalho formal. Seus arquivos são grandes
(~12 M linhas/ano), então a leitura é feita em **chunks** para evitar estouro de memória.

- A partir de 2018: arquivo consolidado `RAIS_VINC_PUB_NORDESTE.7z`
- 2015–2017: arquivos individuais por UF (`AL{ano}.7z`, `BA{ano}.7z`, …)

In [ ]:
RAIS_COLUNAS_ALVO = {
    'Município'              : 'municipio',
    'CNAE 2.0 Subclasse'     : 'cnae_subclasse',
    'CNAE 2.0 Classe'        : 'cnae_subclasse',
    'Vl Remun Média Nom'     : 'remuneracao_media',
    'Qtd Hora Contr'         : 'horas_contratadas',
    'Escolaridade após 2005' : 'grau_instrucao',
    'Sexo Trabalhador'       : 'sexo',
    'Raça Cor'               : 'raca_cor',
    'Vínculo Ativo 31/12'    : 'vinculo_ativo',
}


def _baixar_rais_ano(ano: int, tmpdir: str) -> list:
    """
    Baixa o(s) arquivo(s) RAIS do FTP para um dado ano.
    Tenta primeiro o arquivo consolidado Nordeste; cai para individuais por UF se necessário.
    Retorna lista de paths dos .7z baixados.
    """
    remote_consol = f'{RAIS_FTP_BASE}/{ano}/RAIS_VINC_PUB_NORDESTE.7z'
    local_consol  = Path(tmpdir) / f'RAIS_VINC_PUB_NORDESTE_{ano}.7z'
    try:
        _ftp_download(remote_consol, str(local_consol))
        return [local_consol]
    except error_perm:
        logger.info(f'RAIS {ano}: arquivo consolidado Nordeste não existe, tentando por UF...')
    except Exception as e:
        logger.warning(f'RAIS {ano}: erro consolidado ({e}), tentando por UF...')

    arquivos = []
    for uf_sig in ('AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE'):
        remote_uf = f'{RAIS_FTP_BASE}/{ano}/{uf_sig}{ano}.7z'
        local_uf  = Path(tmpdir) / f'{uf_sig}{ano}.7z'
        try:
            _ftp_download(remote_uf, str(local_uf))
            arquivos.append(local_uf)
        except error_perm:
            logger.warning(f'RAIS {ano}/{uf_sig}: não disponível.')
        except Exception as e:
            logger.warning(f'RAIS {ano}/{uf_sig}: erro download ({e})')
    return arquivos


def _ler_rais_chunked(archive_path: str, extract_dir: str, ano: int):
    """Lê RAIS de um .7z em chunks (500k linhas) para evitar OOM."""
    import py7zr

    with py7zr.SevenZipFile(archive_path, 'r') as z:
        names = z.getnames()
        data_file = next((n for n in names if n.lower().endswith(('.csv', '.txt'))), None)
        if data_file is None:
            raise ValueError(f'Nenhum CSV/TXT em {archive_path}: {names}')
        logger.info(f'Extraindo {data_file}...')
        z.extractall(path=extract_dir)

    file_path = Path(extract_dir) / data_file

    try:
        header_df = pd.read_csv(file_path, sep=';', encoding='latin-1', nrows=0)
    except UnicodeDecodeError:
        header_df = pd.read_csv(file_path, sep=';', encoding='utf-8', nrows=0)

    all_cols = list(header_df.columns)
    usecols, col_map = [], {}
    for orig, target in RAIS_COLUNAS_ALVO.items():
        if orig in all_cols and target not in col_map.values():
            usecols.append(orig)
            col_map[orig] = target

    if not usecols:
        raise ValueError(f'RAIS {ano}: nenhuma coluna conhecida encontrada.')

    chunk_size    = 500_000
    vinculos_aggs = []
    setor_aggs    = []

    try:
        reader = pd.read_csv(file_path, sep=';', encoding='latin-1',
                             usecols=usecols, low_memory=False, chunksize=chunk_size)
    except UnicodeDecodeError:
        reader = pd.read_csv(file_path, sep=';', encoding='utf-8',
                             usecols=usecols, low_memory=False, chunksize=chunk_size)

    for chunk in reader:
        chunk.rename(columns=col_map, inplace=True)

        if 'municipio' in chunk.columns:
            chunk['municipio'] = pd.to_numeric(chunk['municipio'], errors='coerce')
            chunk['uf_cod']    = chunk['municipio'] // 10000
            chunk = chunk[chunk['uf_cod'].isin(UFS_NE_IBGE)]
            chunk['sigla_uf'] = chunk['uf_cod'].map(UF_IBGE_SIGLA)
        else:
            uf_sigla = Path(archive_path).stem[:2].upper()
            chunk['sigla_uf'] = uf_sigla

        if chunk.empty:
            continue

        chunk['ano'] = ano

        if 'vinculo_ativo' in chunk.columns:
            v_ativo = chunk['vinculo_ativo'].astype(str).str.strip().str.upper()
            chunk = chunk[v_ativo.isin({'1', 'S', 'SIM', 'TRUE'})].copy()
        if chunk.empty:
            continue

        if 'remuneracao_media' in chunk.columns:
            chunk['remuneracao_media'] = pd.to_numeric(chunk['remuneracao_media'], errors='coerce')
        else:
            chunk['remuneracao_media'] = pd.NA

        chunk['remuneracao_obs']   = chunk['remuneracao_media'].notna().astype(int)
        chunk['remuneracao_total'] = chunk['remuneracao_media'].fillna(0)

        agg1 = chunk.groupby(['ano', 'sigla_uf'], as_index=False).agg(
            vinculos_ativos=('ano', 'count'),
            remuneracao_total=('remuneracao_total', 'sum'),
            remuneracao_obs=('remuneracao_obs', 'sum'),
        )
        vinculos_aggs.append(agg1)

        if 'cnae_subclasse' in chunk.columns:
            chunk['divisao_cnae'] = chunk['cnae_subclasse'].astype(str).str[:2]
            agg2 = chunk.groupby(['ano', 'sigla_uf', 'divisao_cnae'], as_index=False).agg(
                vinculos_ativos=('ano', 'count'),
                remuneracao_total=('remuneracao_total', 'sum'),
                remuneracao_obs=('remuneracao_obs', 'sum'),
            )
            setor_aggs.append(agg2)

    df_vinculos = pd.DataFrame()
    if vinculos_aggs:
        dv = pd.concat(vinculos_aggs, ignore_index=True)
        dv = dv.groupby(['ano', 'sigla_uf'], as_index=False).agg(
            vinculos_ativos=('vinculos_ativos', 'sum'),
            remuneracao_total=('remuneracao_total', 'sum'),
            remuneracao_obs=('remuneracao_obs', 'sum'),
        )
        dv['remuneracao_media'] = dv['remuneracao_total'] / dv['remuneracao_obs'].replace(0, pd.NA)
        df_vinculos = dv[['ano', 'sigla_uf', 'vinculos_ativos', 'remuneracao_media']]

    df_setor = pd.DataFrame()
    if setor_aggs:
        ds = pd.concat(setor_aggs, ignore_index=True)
        ds = ds.groupby(['ano', 'sigla_uf', 'divisao_cnae'], as_index=False).agg(
            vinculos_ativos=('vinculos_ativos', 'sum'),
            remuneracao_total=('remuneracao_total', 'sum'),
            remuneracao_obs=('remuneracao_obs', 'sum'),
        )
        ds['remuneracao_media'] = ds['remuneracao_total'] / ds['remuneracao_obs'].replace(0, pd.NA)
        df_setor = ds[['ano', 'sigla_uf', 'divisao_cnae', 'vinculos_ativos', 'remuneracao_media']]

    return df_vinculos, df_setor


def coletar_rais_nordeste(ano_inicio: int = PERIODO_INICIO, ano_fim: int = None):
    """Coleta RAIS Nordeste e gera 2 CSVs. Se já existirem no disco, pula o download (cache)."""
    if ano_fim is None:
        ano_fim = PERIODO_FIM - 2

    outputs = [
        RAW_DIR / 'rais' / 'nordeste' / 'rais_vinculos.csv',
        RAW_DIR / 'rais' / 'nordeste' / 'rais_por_setor.csv',
    ]
    if _all_paths_exist(outputs):
        logger.info('RAIS Nordeste: cache encontrado, pulando coleta.')
        return pd.read_csv(outputs[0])

    vinculos_frames, setor_frames, status_rows = [], [], []

    for ano in range(ano_inicio, ano_fim + 1):
        with tempfile.TemporaryDirectory() as tmpdir:
            arquivos = _baixar_rais_ano(ano, tmpdir)
            if not arquivos:
                logger.warning(f'RAIS {ano}: nenhum arquivo disponível.')
                status_rows.append({'fonte': 'rais', 'ano': ano, 'status': 'sem_arquivo'})
                continue
            for arq in arquivos:
                try:
                    df_v, df_s = _ler_rais_chunked(str(arq), tmpdir, ano)
                    if not df_v.empty: vinculos_frames.append(df_v)
                    if not df_s.empty: setor_frames.append(df_s)
                    status_rows.append({'fonte': 'rais', 'ano': ano, 'arquivo': arq.name, 'status': 'ok'})
                except Exception as e:
                    logger.error(f'RAIS {ano} ({arq.name}): erro ({e})')
                    status_rows.append({'fonte': 'rais', 'ano': ano, 'arquivo': arq.name,
                                        'status': 'erro', 'detalhe': str(e)})

    result = None
    if vinculos_frames:
        df_vin = pd.concat(vinculos_frames, ignore_index=True)
        df_vin = df_vin.groupby(['ano', 'sigla_uf'], as_index=False).agg(
            vinculos_ativos=('vinculos_ativos', 'sum'),
            remuneracao_media=('remuneracao_media', 'mean'))
        _save_dataframe(df_vin.sort_values(['ano', 'sigla_uf']),
                        'rais_vinculos', path_parts=['rais', 'nordeste'])
        result = df_vin
    if setor_frames:
        df_set = pd.concat(setor_frames, ignore_index=True)
        df_set = df_set.groupby(['ano', 'sigla_uf', 'divisao_cnae'], as_index=False).agg(
            vinculos_ativos=('vinculos_ativos', 'sum'),
            remuneracao_media=('remuneracao_media', 'mean'))
        _save_dataframe(df_set.sort_values(['ano', 'sigla_uf', 'divisao_cnae']),
                        'rais_por_setor', path_parts=['rais', 'nordeste'])
    _save_status_manifest(status_rows, 'rais_status')
    return result


df_rais = coletar_rais_nordeste()

18:38:27 | INFO | FTP download: ftp.mtps.gov.br/pdet/microdados/RAIS/2015/RAIS_VINC_PUB_NORDESTE.7z
18:38:28 | INFO | RAIS 2015: arquivo consolidado Nordeste não existe, tentando por UF...
18:38:28 | INFO | FTP download: ftp.mtps.gov.br/pdet/microdados/RAIS/2015/AL2015.7z
18:38:40 | INFO | Download concluído: AL2015.7z
18:38:40 | INFO | FTP download: ftp.mtps.gov.br/pdet/microdados/RAIS/2015/BA2015.7z
18:52:04 | INFO | Download concluído: BA2015.7z
18:52:04 | INFO | FTP download: ftp.mtps.gov.br/pdet/microdados/RAIS/2015/CE2015.7z
18:52:45 | INFO | Download concluído: CE2015.7z
18:52:45 | INFO | FTP download: ftp.mtps.gov.br/pdet/microdados/RAIS/2015/MA2015.7z
18:53:01 | INFO | Download concluído: MA2015.7z
18:53:01 | INFO | FTP download: ftp.mtps.gov.br/pdet/microdados/RAIS/2015/PB2015.7z
18:53:18 | INFO | Download concluído: PB2015.7z
18:53:18 | INFO | FTP download: ftp.mtps.gov.br/pdet/microdados/RAIS/2015/PE2015.7z
18:54:05 | INFO | Download concluído: PE2015.7z
18:54:05 | INFO | F

---
## 3. Análise Exploratória

> A partir daqui, assume-se que os CSVs já foram gerados pela pipeline (ou foram
> fornecidos previamente). Altere a variável `BASE` se necessário.

### 3.1 — Carregamento e Unificação das Séries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='Set2', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

BASE = str(RAW_DIR / 'caged' / 'nordeste') + '/'

df_novo          = pd.read_csv(BASE + 'caged_saldo_mensal.csv')
df_antigo        = pd.read_csv(BASE + 'caged_antigo_saldo_mensal.csv')
df_setor         = pd.read_csv(BASE + 'caged_por_setor.csv')
df_perfil        = pd.read_csv(BASE + 'caged_por_perfil.csv')
df_antigo_perfil = pd.read_csv(BASE + 'caged_antigo_por_perfil.csv')

df_mensal = pd.concat([df_antigo, df_novo], ignore_index=True)
df_mensal['data'] = pd.to_datetime(
    df_mensal['ano'].astype(str) + '-' + df_mensal['mes'].astype(str) + '-01')
df_mensal.sort_values(['data', 'sigla_uf'], inplace=True)

df_perfil_total = pd.concat([df_antigo_perfil, df_perfil], ignore_index=True)

print(f'Série mensal unificada: {len(df_mensal)} registros')
print(f'Período: {df_mensal.data.min().strftime("%Y-%m")} a {df_mensal.data.max().strftime("%Y-%m")}')
print(f'Estados: {sorted(df_mensal.sigla_uf.unique())}')

### 3.2 — Panorama Geral — Saldo de Emprego Formal no Nordeste

Evolução do saldo agregado (admissões − desligamentos) mensal para toda a região Nordeste.
A linha escura representa a média móvel de 12 meses.

In [ ]:
ne_mensal = df_mensal.groupby('data').agg(
    saldo=('saldo', 'sum'),
    admissoes=('admissoes', 'sum'),
    desligamentos=('desligamentos', 'sum'),
    salario_medio=('salario_medio', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(16, 6))
colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in ne_mensal['saldo']]
ax.bar(ne_mensal['data'], ne_mensal['saldo'], color=colors, width=25, alpha=0.85)
ax.axhline(y=0, color='black', linewidth=0.8)

ne_mensal['saldo_mm12'] = ne_mensal['saldo'].rolling(12, min_periods=6).mean()
ax.plot(ne_mensal['data'], ne_mensal['saldo_mm12'], color='#2c3e50', linewidth=2.5, label='Média Móvel 12m')

ax.set_title('Saldo de Emprego Formal — Nordeste (2015–2025)', fontweight='bold')
ax.set_ylabel('Saldo (admissões − desligamentos)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.legend()
plt.tight_layout()
plt.show()

### 3.3 — Saldo Acumulado Anual por Estado

Comparação do saldo anual acumulado entre os 9 estados do Nordeste.

In [ ]:
saldo_anual = df_mensal.groupby(['ano', 'sigla_uf'])['saldo'].sum().reset_index()

fig, ax = plt.subplots(figsize=(16, 7))
pivot = saldo_anual.pivot(index='ano', columns='sigla_uf', values='saldo')
pivot.plot(kind='bar', ax=ax, width=0.85)

ax.axhline(y=0, color='black', linewidth=0.8)
ax.set_title('Saldo Anual de Emprego Formal por Estado — Nordeste', fontweight='bold')
ax.set_ylabel('Saldo anual')
ax.set_xlabel('')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.legend(title='UF', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### 3.4 — Heatmap — Saldo Mensal por Estado

Visualização matricial do saldo mensal para identificar padrões sazonais
e choques (ex.: COVID-19 em 2020).

- 🟢 Verde = saldo positivo (mais admissões que desligamentos)
- 🔴 Vermelho = saldo negativo

In [ ]:
df_mensal['ano_mes'] = df_mensal['data'].dt.strftime('%Y-%m')
heatmap_data = df_mensal.pivot_table(index='sigla_uf', columns='ano_mes', values='saldo', aggfunc='sum')

fig, ax = plt.subplots(figsize=(22, 6))
sns.heatmap(
    heatmap_data, cmap='RdYlGn', center=0, ax=ax,
    linewidths=0.1, cbar_kws={'label': 'Saldo', 'shrink': 0.8},
    xticklabels=6
)
ax.set_title('Heatmap — Saldo Mensal de Emprego por Estado (2015–2025)', fontweight='bold')
ax.set_ylabel('')
ax.set_xlabel('')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 3.5 — Evolução do Salário Médio por Estado

Salário médio nominal das movimentações CAGED ao longo do tempo
(média móvel de 6 meses para suavizar a série).

In [ ]:
fig, ax = plt.subplots(figsize=(16, 7))

for uf in sorted(df_mensal.sigla_uf.unique()):
    uf_data = df_mensal[df_mensal.sigla_uf == uf].sort_values('data')
    uf_data['sal_mm6'] = uf_data['salario_medio'].rolling(6, min_periods=3).mean()
    ax.plot(uf_data['data'], uf_data['sal_mm6'], label=uf, linewidth=1.8)

ax.set_title('Salário Médio Nominal — Média Móvel 6 Meses por Estado', fontweight='bold')
ax.set_ylabel('Salário médio (R$)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R$ {x:,.0f}'))
ax.legend(title='UF', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

### 3.6 — Impacto da COVID-19 no Emprego Formal (2020)

Análise do choque no mercado de trabalho durante a pandemia,
com foco nos meses de maior impacto.

In [ ]:
covid_data = ne_mensal[(ne_mensal['data'].dt.year >= 2019) &
                        (ne_mensal['data'].dt.year <= 2021)].copy()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

ax = axes[0]
colors_covid = ['#2ecc71' if v >= 0 else '#e74c3c' for v in covid_data['saldo']]
ax.bar(covid_data['data'], covid_data['saldo'], color=colors_covid, width=25, alpha=0.85)
ax.axhline(y=0, color='black', linewidth=0.8)
ax.axvspan(pd.Timestamp('2020-03-01'), pd.Timestamp('2020-06-30'),
           alpha=0.15, color='red', label='Mar–Jun 2020')
ax.set_title('Saldo Mensal — Nordeste (2019–2021)', fontweight='bold')
ax.set_ylabel('Saldo')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
ax.legend()

ax2 = axes[1]
saldo_2020 = df_mensal[df_mensal['ano'] == 2020].groupby('sigla_uf')['saldo'].sum().sort_values()
ax2.barh(saldo_2020.index, saldo_2020.values,
         color=['#e74c3c' if v < 0 else '#2ecc71' for v in saldo_2020.values])
ax2.axvline(x=0, color='black', linewidth=0.8)
ax2.set_title('Saldo Acumulado em 2020 por Estado', fontweight='bold')
ax2.set_xlabel('Saldo anual')
ax2.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))

plt.tight_layout()
plt.show()

print('\n=== Meses com maior queda em 2020 (Nordeste consolidado) ===')
print(covid_data[covid_data['data'].dt.year == 2020][['data', 'saldo']]
      .sort_values('saldo').head(5).to_string(index=False))

### 3.7 — Saldo por Setor Econômico (CNAE)

Top setores com maiores saldos positivos e negativos nos últimos anos disponíveis.

In [ ]:
CNAE_NOMES = {
    '01': 'Agricultura',
    '10': 'Alimentos',
    '41': 'Construção Civil',
    '47': 'Comércio Varejista',
    '56': 'Alimentação',
    '62': 'TI',
    '68': 'Imóveis',
    '78': 'Seleção/Colocação',
    '84': 'Administração Pública',
    '85': 'Educação',
    '86': 'Saúde',
    '96': 'Serv. Pessoais',
}

ano_ref = df_setor['ano'].max()
setor_ano = df_setor[df_setor['ano'] == ano_ref].groupby('divisao_cnae')['saldo'].sum().sort_values()
setor_ano.index = setor_ano.index.map(lambda x: CNAE_NOMES.get(str(x).zfill(2), f'CNAE {x}'))

top15 = setor_ano.reindex(setor_ano.abs().nlargest(15).index)

fig, ax = plt.subplots(figsize=(14, 8))
colors_setor = ['#2ecc71' if v >= 0 else '#e74c3c' for v in top15.values]
ax.barh(top15.index, top15.values, color=colors_setor)
ax.axvline(x=0, color='black', linewidth=0.8)
ax.set_title(f'Saldo por Setor CNAE — Nordeste ({ano_ref})', fontweight='bold')
ax.set_xlabel('Saldo (admissões − desligamentos)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}k'))
plt.tight_layout()
plt.show()

### 3.8 — Perfil das Admissões: Sexo e Escolaridade

Distribuição dos trabalhadores admitidos por sexo e grau de instrução.

In [ ]:
GRAU_INSTRUCAO = {
    1: 'Analfabeto',
    2: 'Até 5ª Inc.',
    3: '5ª Comp.',
    4: '6ª a 9ª Inc.',
    5: 'Fund. Comp.',
    6: 'Médio Inc.',
    7: 'Médio Comp.',
    8: 'Sup. Inc.',
    9: 'Sup. Comp.',
    10: 'Mestrado',
    11: 'Doutorado',
}
SEXO = {1: 'Masculino', 2: 'Feminino'}

perfil_ano = (df_perfil_total.groupby(['sexo', 'grau_instrucao'])
              .agg(admissoes=('admissoes', 'sum')).reset_index())
perfil_ano['sexo_nome'] = perfil_ano['sexo'].map(SEXO).fillna('Outro')
perfil_ano['grau_nome'] = perfil_ano['grau_instrucao'].map(GRAU_INSTRUCAO).fillna('N/D')

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

sexo_tot = perfil_ano.groupby('sexo_nome')['admissoes'].sum().sort_values(ascending=True)
axes[0].barh(sexo_tot.index, sexo_tot.values, color=['#3498db', '#e74c3c'])
axes[0].set_title('Total de Admissões por Sexo (2015–2025)', fontweight='bold')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

grau_tot = perfil_ano.groupby('grau_nome')['admissoes'].sum().sort_values(ascending=True)
axes[1].barh(grau_tot.index, grau_tot.values, color='#3498db')
axes[1].set_title('Total de Admissões por Grau de Instrução (2015–2025)', fontweight='bold')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

plt.tight_layout()
plt.show()

---
## Conclusão

Este notebook demonstrou o fluxo completo de trabalho com dados do CAGED e da RAIS:

1. **Extração** — download dos microdados via FTP do MTE/PDET, descompactação de arquivos `.7z` e tratamento de encoding (latin-1/UTF-8) e mojibake.
2. **Transformação** — normalização dos nomes de colunas (que variam entre os anos), filtro geográfico para o Nordeste e agregação em CSVs menores.
3. **Análise** — visualizações que revelaram:
   - O choque da COVID-19 em 2020 (saldo negativo histórico em mar–jun).
   - A recuperação pós-pandemia a partir de 2021.
   - Os setores com maior dinâmica de contratação no Nordeste.
   - O perfil predominante dos trabalhadores formais (escolaridade e sexo).

### Próximos passos sugeridos
- Cruzar CAGED com RAIS para análise de rotatividade.
- Incorporar dados do PIB estadual para correlacionar emprego e atividade econômica.
- Usar `statsmodels` ou `prophet` para previsão de série temporal do saldo de emprego.